# Repair time and chunk durability

I use this notebook to connect measured repair durations to an intentionally small probability model. I built this to learn the trade-offs. The model is not a reliability claim about my laptop, Docker or a real deployment.

Backblaze publishes daily drive observations and aggregate counts in [Drive Stats](https://www.backblaze.com/cloud-storage/resources/hard-drive-test-data). My checked-in assumption file records the published 2025 aggregate: 4,317 failures over 115,638,676 drive-days. I calculate the failure intensity rather than treating the rounded AFR as a probability. Backblaze asks for attribution and prohibits selling the data itself. I do not redistribute its dataset here.

The model assumes independent, constant exponential node failure times, one serial repair worker per chunk, exponential repair durations, immediate replacement capacity, and a healthy initial chunk. A state is the number of missing holders. Losing holder m+1 is absorbing. In state i, failure rate is (k+m-i)λ and repair rate is μ=1/T. Three replicas use k=1,m=2. Correlated failures, a limited spare pool, silent damage detection, queues across objects and coordinator downtime need separate treatment. Six nodes after two permanent failures violate the replacement-capacity assumption.


In [ ]:
from pathlib import Path
import csv, json, sys
import matplotlib.pyplot as plt
root = Path.cwd()
if root.name == 'durability': root = root.parent
sys.path.insert(0, str(root / 'durability'))
from model import annual_loss
assumptions = json.loads((root / 'durability/backblaze-2025.json').read_text())
failure_rate = assumptions['failures'] * 365.25 / assumptions['drive_days']
assumptions, failure_rate


## Measurements

Run `python3 scripts/e2e.py --measure` first. These observations come from actual Java processes with mTLS and H2, not containers or a production cluster. `repair-one` times a synchronous repair of one corrupt shard through target verification and the metadata update. It includes CLI startup and network calls. I add the default 30-second heartbeat timeout plus repair delay to illustrate detection overhead. The corruption scan has its own interval. This timing is only an approximation of the exponential per-holder mean assumed by the model; it is not a measured distribution. Multi-chunk node repairs and `repair-node` observations should be studied separately because the worker repairs several shards.


In [ ]:
measurements = root / 'measurements/runs.csv'
if not measurements.exists():
    raise FileNotFoundError('Collect real observations with scripts/e2e.py --measure first')
with measurements.open(newline='') as source:
    rows = [r for r in csv.DictReader(source) if int(r['exit_code']) == 0]
repairs = [float(r['elapsed_seconds']) for r in rows if r['label'] == 'repair-one']
if not repairs: raise ValueError('No successful single-shard repair observations')
repair_mean = sum(repairs) / len(repairs)
detection_seconds = 30
window = repair_mean + detection_seconds
print({'samples': len(repairs), 'mean_manual_seconds': repair_mean,
       'assumed_detection_seconds': detection_seconds, 'model_mean_seconds': window})


In [ ]:
# This grid is a sensitivity experiment, not additional repair measurements.
times = sorted(set([1, 10, 30, 60, 300, 3600, 86400, window]))
fig, ax = plt.subplots(figsize=(7,4), layout='constrained')
for name,k,m in [('3+2',3,2),('4+2',4,2),('3x replication',1,2)]:
    ax.loglog(times, [annual_loss(k,m,failure_rate,t) for t in times], 'o-', label=name)
ax.axvline(window, color='gray', linestyle=':', label='measured repair + assumed detection')
ax.set(xlabel='Mean repair window per holder (s)', ylabel='Model annual chunk loss probability')
ax.grid(alpha=.3); ax.legend()
fig.savefig(root / 'measurements/durability.png', dpi=180)


In [ ]:
fig, ax = plt.subplots(figsize=(7,4), layout='constrained')
parity = list(range(5))
for k in [1,3,4]:
    ax.semilogy(parity, [annual_loss(k,m,failure_rate,window) for m in parity], 'o-', label=f'k={k}')
ax.set(xlabel='Parity holders', ylabel='Model annual chunk loss probability', xticks=parity)
ax.grid(alpha=.3); ax.legend()
fig.savefig(root / 'measurements/parity.png', dpi=180)


In [ ]:
# Inspect successful throughput and peak memory without hiding failed runs.
for metric in ['throughput', 'memory', 'repair']:
    from subprocess import run
    run([sys.executable, str(root/'durability/plot.py'), metric, str(measurements),
         '--output', str(root/f'measurements/{metric}.png')], check=True)


## What I would change at ten times the scale

I would keep metadata transactional, paginate the repair queue with persistent retry deadlines, measure queue delay and maintain fair bandwidth budgets. I would then add coordinator replication or a consensus-backed metadata service, failure domains in placement, independent backups, key recovery and rotation, and explicit metadata rollback protection. Before trusting probabilities I would compare repair-time distributions, use correlated outage scenarios, and model limited replacement capacity. A tiny probability from the independent model does not establish durability.
